In [0]:
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 

In [0]:
dbutils.library.restartPython()

In [0]:

import json
from common_utils.data_quailtychecks import dataquailty
from pyspark.sql.functions import trim
import pyspark.sql.functions as F
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import to_date
from common_utils.logging import get_logger
logger = get_logger("Data Engineering")


# =========================================
# 1. CONFIG PATH
# =========================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")
print(config_path)


# =========================================
# 2. READ CONFIGURATION FILE
# =========================================

with open(config_path, mode='r') as f:
    config = json.load(f)


# =========================================
# 3. GET SOURCE PATH
# =========================================

source_path = config['source']['source_table']
people_id=config['columns']["people_id"]
first_name=config['columns']["first_name"]
surname=config['columns']["surname"]
job_role=config['columns']["job_role"]
employment_type=config['columns']["employment_type"]
team=config['columns']["team"]
hire_date=config['columns']["hire_date"]
distinct_columns = [
    employment_type,
    team,
    job_role
]


# =========================================
# 4. READ SOURCE DATA
# =========================================

df = spark.read.table(source_path)
df.display()
# =========================================
# 4. Data Quailty Checks
# =========================================
dataquailty(df,people_id,distinct_columns)

# =========================================
# 5. Writing into Silver Layer
# =========================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS group_project.silver"
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config['target']['target_table'])
)

logger.info(
    "Successfully loaded into Silver",

)
